In [1]:
from datetime import datetime, timedelta
from utils.spark_session import get_sparkSession 
from utils.utilities import generate_date
from utils.s3_aws import archive_landing

In [2]:
from pyspark.sql.functions import current_timestamp, lit

In [3]:
spark = get_sparkSession("Load bronze.fact_customer_events")

In [4]:
print("SPARK-APP : spark-UI - " + spark.sparkContext.uiWebUrl)

SPARK-APP : spark-UI - http://ba7b2bbb9aec:4040


In [5]:
from utils.load_controller import insert_control_logs, get_max_loadTimestamp

In [6]:
_schema_name = "bronze"
_table_name = "fact_customer_events"
_fullname = f"{_schema_name}.{_table_name}"
_filename = "dim_customer_events.json"
_file_path = f"s3a://ecommerce-pooja/pipeline/landing/customer_events/{_filename}"
_script_name = "raw_fact_customer_events.ipynb"

print(f"SPARK-APP : Loading started for {_fullname}")

SPARK-APP : Loading started for bronze.fact_customer_events


In [7]:
#spark config

spark.conf.set("spark.sql.shuffle.partitions",16)
spark.conf.set("spark.sql.parquet.mergeSchema", True)


In [8]:
#Reading json file & creating dataframe

df = spark.read.format("text") \
          .option("multiline", False) \
          .load(_file_path)

loaded_count = df.count()

In [9]:
df.show(truncate=False)
df.printSchema()
print(f"SPARK-APP : Rows entered : {loaded_count}")

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        |
+---------------------------------------

In [10]:

df_ld = df.withColumn("created_on", current_timestamp()) \
       .withColumn("created_by", lit(_script_name)) \
       .withColumn("modified_on", current_timestamp()) \
       .withColumn("modified_by", lit(_script_name)) \
       .withColumn("source_file", lit("dim_customer_events.json"))

In [11]:
df_ld.show(10)

+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|               value|          created_on|          created_by|         modified_on|         modified_by|         source_file|
+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|{"event_id":"EVT0...|2026-01-29 00:24:...|raw_fact_customer...|2026-01-29 00:24:...|raw_fact_customer...|dim_customer_even...|
|{"event_id":"EVT0...|2026-01-29 00:24:...|raw_fact_customer...|2026-01-29 00:24:...|raw_fact_customer...|dim_customer_even...|
|{"event_id":"EVT0...|2026-01-29 00:24:...|raw_fact_customer...|2026-01-29 00:24:...|raw_fact_customer...|dim_customer_even...|
|{"event_id":"EVT0...|2026-01-29 00:24:...|raw_fact_customer...|2026-01-29 00:24:...|raw_fact_customer...|dim_customer_even...|
+--------------------+--------------------+--------------------+--------------------+-------------------

In [12]:
max_timestamp = get_max_loadTimestamp(spark, _schema_name, _table_name)

_data = []
if max_timestamp != '1900-01-01 00:00:00.000000':
    df_ld.write \
      .format("delta") \
      .mode("append") \
      .saveAsTable(_fullname)
    
    _data.append([_schema_name, _schema_name, _table_name, "delta-load", "append", str(datetime.now()), "success", loaded_count, _script_name, _script_name])
    insert_control_logs(spark, _data)
    
else:
    df_ld.write \
      .format("delta") \
      .mode("overwrite") \
      .saveAsTable(_fullname)
    
    _data.append([_schema_name, _schema_name, _table_name, "full-load", "overwrite", str(datetime.now()), "success", loaded_count, _script_name, _script_name])
    insert_control_logs(spark, _data)
    
print(f"SPARK-APP: Data written to {_schema_name}.{_table_name} and load_controller")

SPARK-APP: Data written to bronze.fact_customer_events and load_controller


In [13]:
spark.sql(f"select * from gold.load_controller where schema_name = '{_schema_name}' and table_name = '{_table_name}' order by created_on desc").show()

+------+-----------+--------------------+---------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+
| layer|schema_name|          table_name|load_type|write_mode|      load_timestamp|load_status|loaded_count|          created_on|          created_by|         modified_on|         modified_by|
+------+-----------+--------------------+---------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+
|bronze|     bronze|fact_customer_events|full-load| overwrite|2026-01-29 00:25:...|    success|           4|2026-01-29 00:25:...|raw_fact_customer...|2026-01-29 00:25:...|raw_fact_customer...|
+------+-----------+--------------------+---------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+



In [14]:
from delta import DeltaTable

dt = DeltaTable.forName(spark,f"{_schema_name}.{_table_name}")

dt.history().limit(1).select("version","operationMetrics.numOutputRows").show(truncate=False)

+-------+-------------+
|version|numOutputRows|
+-------+-------------+
|0      |4            |
+-------+-------------+



In [15]:
#Archive files

if archive_landing(_filename, "customer_events"):
    print(f"SPARK-APP: File {_filename} archived")
else:
    print(f"SPARK-APP: Error while archiving {_filename} from landing")

SPARK-APP: File dim_customer_events.json archived


In [16]:
spark.stop()